# Residual SAC on SmolVLA, LIBERO-10 task 6 (Kaggle, 2 x T4)

Pilot of `rl_based_tuning/` (see its README): a small residual head trained with SAC on top of the frozen
base policy `lerobot/smolvla_libero` (57.0% on LIBERO-10; 14/30 on task 6 in
`week6/smolvla_libero_lerobot_eval_kaggle/`). Training uses init states 10-49; evaluation uses init states 0-9
x seeds 0, 1, 2, base and residual on the same episodes.

Set `STAGE` in step 1, then **Save Version > Save & Run All (Commit)** (GPU T4 x2, Internet on):

0. `"diagnose"` (about 2-3 h, added after pilot 1): the base policy alone with soft and hard resets, and with
   the untrained head's exploration noise at pilot 1's uniform 0.05 and at the per-dimension 0.15 std, each on
   the same 20 training init states and episode seeds (paired). Shows whether the environment or the noise
   alone explains pilot 1's drop (`week6/sac_residual_task6_pilot1/`).

1. `"verify"` (about 2 h): unit tests, a short training and evaluation smoke run, and the **harness check**: the
   base policy through the RL evaluation path on task 6, compared with lerobot-eval's 14/30 on the same init
   states. If they disagree, stop: RL numbers from a broken harness mean nothing.
2. `"train"` (one or more 12 h sessions): one training run per seed in `TRAIN_SEEDS`, one per GPU. Each stops
   starting episodes after `TIME_LIMIT_HOURS`; to continue, add the previous version's output as input
   (**Add Input > Your Work >** this notebook) and run again with the same settings.
3. `"eval"`: each trained residual against the base on the held-out episodes, with an exact McNemar test.

Every stage reads the previous version's output if it was added as input, so the stages chain through versions.

## 1. Settings

In [ ]:
import csv, glob, json, os, re, shutil, subprocess, sys, time
from pathlib import Path

STAGE = "diagnose"                 # "verify", "diagnose", then "train" (repeat until done), then "eval"
RUN_NAME = "pilot2"                # training/eval folder; pilot 1 (uniform 0.05 residual, no warm-up) was seed_0/seed_1

REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "main"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

OUT_NAME = "sac_residual_task6"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"  # results and checkpoints, kept as the version's output

TASK_ID = 6
TRAIN_SEEDS = [0, 1]               # one training run per GPU
NUM_EPISODES = 100                 # training episodes per seed (about 7-8 min each on a T4)
TIME_LIMIT_HOURS = 10.0            # no new training episode after this; Kaggle stops a session at 12 h
EVAL_SEEDS = [0, 1, 2]
EVAL_INIT_STATES = "0-9"
DIAGNOSTIC_EPISODES = 20           # distinct training init states per diagnostic condition
GB_PER_JOB = 7                     # base policy + 2 GB replay buffer per training process

# Base policy: lerobot/smolvla_libero at the revision evaluated in week6/smolvla_libero_lerobot_eval_kaggle.
POLICY_REPO, POLICY_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

A separate virtual environment with the package versions of the local `lerobot_env`. If the GPU driver is
too old for the CUDA 13 build of PyTorch, the same PyTorch version is installed in a CUDA 12 build instead
and the choice is printed. Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base-policy snapshot

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", OMP_NUM_THREADS="1", PYTHONPATH=REPO, PYTHONFAULTHANDLER="1",
               WANDB_MODE="offline", WANDB_DIR=f"{OUT_ROOT}/wandb")   # W&B logs kept in the output; sync later

sh(f"""{PY} -c "
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed; this notebook needs GPU rendering"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)

# Pinned base-policy snapshot, downloaded once; every process loads it from this folder.
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{POLICY_REPO}', revision='{POLICY_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
TRAIN = f"{REPO}/rl_based_tuning/train_online_sac.py"
BASE_ARGS = f"--task_id {TASK_ID} --policy_path {POLICY_PATH}"   # n_action_steps=1 and the rename map are defaults
os.makedirs(OUT_ROOT, exist_ok=True)
print("base policy:", POLICY_PATH)

## 6. Helpers

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=60, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once. Exit code 3 means 'call again' and the
    command is restarted (evaluation runs at most 10 episodes per process); returns {name: final exit code}."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            print(time.strftime("%H:%M"), "started", name, "on GPU", gpu, flush=True)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3 and name.startswith("eval"):
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "finished with exit code", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            for name in running:
                tail = [l for l in open(f"{OUT_ROOT}/logs/{name}.log").read().splitlines() if l.startswith("[")][-1:]
                print(time.strftime("%H:%M"), name, tail, flush=True)
        time.sleep(poll_s)
    return codes


def merge_csv(paths, out):
    rows = [r for p in paths if os.path.exists(p) for r in csv.DictReader(open(p))]
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["task_id", "seed", "init_state", "success", "steps"])
        w.writeheader()
        w.writerows(rows)
    return rows


def eval_jobs(policies, out_dir, extra=""):
    """One evaluation process per eval seed (they run in parallel), each with its own output folder."""
    return {f"eval_{Path(out_dir).name}_s{s}": (
        f"cd {REPO} && {PY} -u {TRAIN} --eval {BASE_ARGS} --eval_policies {policies} --eval_seeds {s} "
        f"--eval_init_states {EVAL_INIT_STATES} --eval_output_dir {out_dir}/seed{s} {extra}", i % 2)
        for i, s in enumerate(EVAL_SEEDS)}


sys.path.insert(0, f"{REPO}/baseline_performance")
from compare_per_state import compare, format_report


def episodes(rows, task=TASK_ID):
    return {(task, int(r["seed"]), int(r["init_state"])): int(r["success"]) for r in rows if int(r["task_id"]) == task}


os.makedirs(f"{OUT_ROOT}/logs", exist_ok=True)

## 7. Resume from an earlier version's output

In [ ]:
# Copy everything from an earlier version's output (added as input) so this stage builds on it.
for prev in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True)):
    print("copying", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
print(sorted(os.listdir(OUT_ROOT)))

## 8. Stage `verify`: smoke run and harness check

In [ ]:
if STAGE == "verify":
    # 1. Smoke run: 2 short training episodes with SAC updates, then a paired evaluation with resume.
    smoke = f"{OUT_ROOT}/_smoke"
    sh(f"cd {REPO} && {PY} -u {TRAIN} {BASE_ARGS} --num_episodes 2 --max_steps 30 --preflight_episodes 1 "
       f"--preflight_min_success 0 --batch_size 8 --buffer_size 200 --residual_image_size 64 "
       f"--checkpoint_dir {smoke} --checkpoint_interval 1", env=RUN_ENV)
    for _ in range(4):
        r = sh(f"cd {REPO} && {PY} -u {TRAIN} --eval {BASE_ARGS} --checkpoint_dir {smoke} --eval_seeds 0 "
               f"--eval_init_states 0,1 --max_steps 30 --max_episodes_per_process 1", check=False, env=RUN_ENV)
        if r.returncode != 3:
            break
    assert r.returncode == 0, "smoke evaluation failed"

    # 2. Harness check: the base policy through the RL evaluation path on task 6, init states 0-9 x 3 seeds.
    codes = run_jobs(eval_jobs("base", f"{OUT_ROOT}/eval_base"), max_parallel=3)
    assert all(c == 0 for c in codes.values()), codes
    mine = merge_csv([f"{OUT_ROOT}/eval_base/seed{s}/per_episode_base.csv" for s in EVAL_SEEDS],
                     f"{OUT_ROOT}/eval_base/per_episode_base.csv")
    ref = list(csv.DictReader(open(f"{REPO}/week6/smolvla_libero_lerobot_eval_kaggle/per_episode.csv")))
    report = format_report(compare(episodes(ref), episodes(mine)), "lerobot-eval (week6)", "RL evaluation path")
    open(f"{OUT_ROOT}/harness_check.txt", "w").write(report)
    print(report)
    print("PASS if the success counts are close and the McNemar p-value is not small (say p > 0.05).")
else:
    print("stage is", STAGE)

## 9. Stage `diagnose`: is it the environment or the exploration noise?

In [ ]:
if STAGE == "diagnose":
    conds = {"base_soft": "--diagnostic base",
             "base_hard": "--diagnostic base --train_hard_reset 1",
             "noise_pilot1": "--diagnostic noise --residual_scale 0.05",
             "noise_new": "--diagnostic noise"}  # per-dimension 0.15 std (the default)
    jobs = {f"diag_{n}": (f"cd {REPO} && {PY} -u {TRAIN} {BASE_ARGS} --seed 0 {a} --diagnostic_episodes "
                          f"{DIAGNOSTIC_EPISODES} --checkpoint_dir {OUT_ROOT}/diagnostic/{n}", i % 2)
            for i, (n, a) in enumerate(conds.items())}
    codes = run_jobs(jobs, max_parallel=4)
    assert all(c == 0 for c in codes.values()), codes
    res = {}
    for n, a in conds.items():
        mode = a.split()[1]
        res[n] = {(TASK_ID, 0, int(r["init_state"])): int(r["success"])
                  for r in csv.DictReader(open(f"{OUT_ROOT}/diagnostic/{n}/diagnostic_{mode}.csv"))}
    lines = [f"{n}: {sum(v.values())}/{len(v)}" for n, v in res.items()]
    for n in ("base_hard", "noise_pilot1", "noise_new"):
        lines.append(format_report(compare(res["base_soft"], res[n]), "base_soft", n))
    report = "\n".join(lines)
    open(f"{OUT_ROOT}/diagnostic/summary.txt", "w").write(report)
    print(report)
    print("Reading: base_soft near 40% (LIBERO-CTRL 16/40 on these init states) clears the environment; noise_pilot1 "
          "far below base_soft means pilot 1's noise alone broke the policy; noise_new should match base_soft.")
else:
    print("stage is", STAGE)

## 10. Stage `train`

In [ ]:
if STAGE == "train":
    import psutil
    jobs = {f"train_{RUN_NAME}_seed{s}": (
        f"cd {REPO} && {PY} -u {TRAIN} {BASE_ARGS} --seed {s} --num_episodes {NUM_EPISODES} "
        f"--checkpoint_dir {OUT_ROOT}/{RUN_NAME}/seed_{s} --eval_init_states {EVAL_INIT_STATES} --resume_training "
        f"--time_limit_hours {TIME_LIMIT_HOURS} --preflight_episodes 3 --preflight_min_success 0.2", i % 2)
        for i, s in enumerate(TRAIN_SEEDS)}
    avail_gb = psutil.virtual_memory().available / 1e9
    assert avail_gb > GB_PER_JOB * len(jobs), f"only {avail_gb:.0f} GB RAM available"
    codes = run_jobs(jobs, max_parallel=len(jobs))
    print(codes, "(0 = all episodes done, 3 = time limit: add this output as input and run again)")
    for s in TRAIN_SEEDS:
        lines = [l for l in open(f"{OUT_ROOT}/logs/train_{RUN_NAME}_seed{s}.log").read().splitlines() if l.startswith("[train] ep")]
        print(f"seed {s}: {len(lines)} episodes this session; last: {lines[-3:]}")
else:
    print("stage is", STAGE)

## 11. Stage `eval`: residual vs base on the same episodes

In [ ]:
if STAGE == "eval":
    base_csv = f"{OUT_ROOT}/eval_base/per_episode_base.csv"
    assert os.path.exists(base_csv), "run the verify stage first (it evaluates the base policy)"
    base_rows = list(csv.DictReader(open(base_csv)))
    jobs = {}
    for s in TRAIN_SEEDS:
        ckpt = f"{OUT_ROOT}/{RUN_NAME}/seed_{s}"
        for es in EVAL_SEEDS:  # reuse the base episodes, so only the residual runs
            d = f"{ckpt}/eval/seed{es}"
            os.makedirs(d, exist_ok=True)
            shutil.copy(f"{OUT_ROOT}/eval_base/seed{es}/per_episode_base.csv", d)
        jobs.update({k.replace("eval_", f"eval_{RUN_NAME}_train{s}_"): v for k, v in
                     eval_jobs("base,residual", f"{ckpt}/eval", extra=f"--checkpoint_dir {ckpt}").items()})
    codes = run_jobs(jobs, max_parallel=4)
    assert all(c == 0 for c in codes.values()), codes
    pooled_b, pooled_r = {}, {}
    for s in TRAIN_SEEDS:
        res = merge_csv([f"{OUT_ROOT}/{RUN_NAME}/seed_{s}/eval/seed{es}/per_episode_residual.csv" for es in EVAL_SEEDS],
                        f"{OUT_ROOT}/{RUN_NAME}/seed_{s}/eval/per_episode_residual.csv")
        b, r = episodes(base_rows), episodes(res)
        report = format_report(compare(b, r), "base", f"residual (training seed {s})")
        open(f"{OUT_ROOT}/{RUN_NAME}/seed_{s}/eval/comparison_base_vs_residual.txt", "w").write(report)
        print(report)
        pooled_b.update({(k[0], 100 * s + k[1], k[2]): v for k, v in b.items()})
        pooled_r.update({(k[0], 100 * s + k[1], k[2]): v for k, v in r.items()})
    report = format_report(compare(pooled_b, pooled_r), "base", "residual, all training seeds")
    open(f"{OUT_ROOT}/{RUN_NAME}/comparison_pooled.txt", "w").write(report)
    print(report)
else:
    print("stage is", STAGE)

## 12. Files for the repo

In [ ]:
# Small files for the repo (no checkpoints, replay buffers or videos); the full folder stays in the Output tab.
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/{OUT_NAME}_results.zip . -x '*.pt' '*.npz' '*/videos/*' '*.mp4' 'wandb/*'")
sh(f"du -h /kaggle/working/{OUT_NAME}_results.zip; du -sh {OUT_ROOT}")